In [1]:
import qi
import os
import sys
import time
import threading

from naoqi import ALProxy

# Bot's IP address and port
ip = '10.104.23.217'
port = 9559

In [2]:
file_manager = ALProxy("ALFileManager", ip, port)
shared_folder_name = file_manager.getUserSharedFolderPath()
shared_folder_name

'/home/nao/'

## Transfer and Receiving File Functions

In [64]:
!pip install paramiko
import paramiko
def transfer_file(remote_path="/home/nao/microphones/recording.wav", local_path="recordings/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.get(remote_path, local_path)

    sftp.close()
    ssh.close()
    print("File transfered")

def receive_file(local_path="recordings/recording.wav", remote_path="/home/nao/microphones/recording.wav"):
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(ip, username='nao', password='nao')

    sftp = ssh.open_sftp()
    sftp.put(local_path, remote_path)

    sftp.close()
    ssh.close()

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


## Audio Recording

In [50]:
def eye_color(status="ListenOn", rgb = None):
    leds = ALProxy("ALLeds", ip, port)

    if rgb is None:
        if status == "ListenOn":
            leds.fadeRGB("AllLeds", "yellow", 1)
        elif status == "SpeechDetected":
            leds.fadeRGB("AllLeds", "green", 8)
        elif status == "EndOfProcess":
            leds.fadeRGB("AllLeds", "white", 1)

    else:
        leds.fadeRGB("Face", rgb, 1)


# Callback function for timer
def timer_callback(timer_cb):
    print("Timer reached")
    timer_cb.set()

# Records Audio when speech is detected
def record_audio(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    speech_recogniser = ALProxy("ALSpeechRecognition", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)

    speech_recogniser.subscribe("speech_recognition")
    speech_recogniser.setLanguage("English")
    vocabulary = ["pepper", "hello"]
    speech_recogniser.setVocabulary(vocabulary, False)

    # Stop recording when speech stops being detected or timer is reached
    timer_cb = threading.Event()
    if timer is not None:
        threading.Timer(timer, timer_callback, timer_cb).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Print status
        if debug:
            print(status)

        # Start recording when speech is detected
        if status == "SpeechDetected":
            eye_color(status)
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            break

    if timer_cb.is_set():
        return

    while True:
        time.sleep(0.5)
        status = memory.getData("ALSpeechRecognition/Status")

        # Stop recording when speech stops being detected
        if status == "EndOfProcess":
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            eye_color(status)
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            recorder.stopMicrophonesRecording()
            speech_recogniser.unsubscribe("speech_recognition")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "White", 1)
            break


def record_audio_sd(timer=None, path_name="/home/nao/microphones/recording.wav", debug=False):
    path_name = os.path.join(path_name)
    # Create Connection Proxies
    recorder = ALProxy("ALAudioRecorder", ip, port)
    sound_detector = ALProxy("ALSoundDetection", ip, port)
    memory = ALProxy("ALMemory", ip, port)
    
    timer_cb = threading.Event()

    sound_detector.subscribe("sound_detector")
    sound_detector.setParameter("Sensitivity", 0.8)

    # Record audio when sound is detected
    if timer is not None:
        threading.Timer(timer, timer_callback, [timer_cb]).start()

    # Start recording
    while True:
        time.sleep(0.5)
        status = memory.getData("SoundDetected")

        # Print status
        if debug:
            print(status)

        # Start recording when sound is detected
        if status is not None:
            print("Recording is starting...")
            recorder.startMicrophonesRecording(path_name, "wav", 16000, (0,0,1,0))
            eye_color("SpeechDetected")
            break

        # Stop recording when timer is reached
        if timer_cb.is_set():
            break

    if timer_cb.is_set():
        return
    
    sound_detector.setParameter("Sensitivity", 0.7)
    while True:
        time.sleep(3)
        status = memory.getData("SoundDetected")

        if debug:
            print("Status: ", status)

        # Stop recording when sound stops being detected
        if status is None:
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "white", 1)
            break

        # Stop recording when timer is reached
        if timer_cb:
            print("Recording is stopping...")
            recorder.stopMicrophonesRecording()
            sound_detector.unsubscribe("sound_detector")
            leds = ALProxy("ALLeds", ip, port)
            leds.fadeRGB("AllLeds", "white", 1)
            break

## Speech to Text

In [ ]:
!pip install speechrecognition==3.8.1
import speech_recognition as sr
def convert_wav_to_text(audio_path):
    r = sr.Recognizer()
    with sr.AudioFile(audio_path) as source:
        audio_data = r.record(source)
    try:
        text = r.recognize_google(audio_data)
        return text
    except sr.UnknownValueError:
        return "Speech recognition could not understand audio"
    except sr.RequestError as e:
        return "Could not request results from Google Speech Recognition service: " + str(e)
    except Exception as e:
        return "An error occurred during speech recognition: " + str(e)

DEPRECATION: Python 2.7 reached the end of its life on January 1st, 2020. Please upgrade your Python as Python 2.7 is no longer maintained. A future version of pip will drop support for Python 2.7. More details about Python 2 support in pip, can be found at https://pip.pypa.io/en/latest/development/release-process/#python-2-support
     |████████████████████████████████| 32.8 MB 6.7 MB/s eta 0:00:011
You should consider upgrading via the '/usr/local/bin/python -m pip install --upgrade pip' command.


## Testing

In [48]:
recorder = ALProxy("ALAudioRecorder", ip, port)
recorder.stopMicrophonesRecording()

In [65]:
record_audio_sd(timer=15, debug=False)
transfer_file()
# Sam, put your code here, also your functions here too

[[6165, 0, 1.0, 0]]
Recording is starting...
('Status: ', [[261, 0, 1.0, 0]])
Recording is stopping...
File transfered
Timer reached


In [60]:
result = convert_wav_to_text("recordings/recording.wav")

In [61]:
result

u"yes just not know what it means I'll be honest with you quite frankly I do not either"

In [ ]:
# Record Video
def record_video(timer=10):
    video_recorder = ALProxy("ALVideoRecorder", ip, port)
    video_recorder.startRecording("/home/nao/recordings/cameras", "test_video")
    time.sleep(timer)
    videoInfo = video_recorder.stopRecording()

    print('Video was saved on the robot: ', videoInfo[1])
    print('Total number of frames: ', videoInfo[0])

('Video was saved on the robot: ', '/home/nao/recordings/cameras/test_video.avi')
('Total number of frames: ', 73)


**Method 2**

In [ ]:
!pip install pyftplib
import pyftplib
from pyftplib import FTPServer
from pyftplib.handlers import FTPHandler
from pyftplib.authorizers import DummyAuthorizer
import threading

def ftp_transfer_file():
    remote_path = "/home/nao/microphones/recording.wav"
    authorizer = DummyAuthorizer()
    authorizer.add_user("nao", "nao", remote_path, perm="elradfmw")
    handler = FTPHandler
    handler.authorizer = authorizer
    server = FTPServer((ip, 21), handler)
    server_thread = threading.Thread(target=server.serve_forever)
    server_thread.daemon = True
    server_thread.start()

# Tablet

In [56]:
tablet_service = ALProxy("ALTabletService", ip, port)
tablet_service.wakeUp()
tablet_service.resetTablet()
tablet_service.enableWifi()

In [57]:
tablet_service.getWifiStatus()

'DISCONNECTED'

In [58]:
# tablet_service.configureWifi("ACpg2DpVU3rCo)z5", "Deakin IoT", "WPA2-Personal")
tablet_service.connectWifi("Deakin IoT")

True

In [52]:
tablet_service.getWifiStatus()

'DISCONNECTED'

In [32]:
# Put code for tablet web view here
